# Prediction analysis

Analysis of the final prediction files by language and model.


## Load and summarize

Load the final prediction files and show only the dataset coverage: language, model, and number of evaluated questions.


In [ ]:
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd() / "data"

files = [
    path for path in sorted(DATA_DIR.glob("prediction_*.csv")) # add EXAMPLE_ for testing with EXAMPLE files
]

language_names = {"en": "English", "sw": "Swahili"}
frames = []
for path in files:
    parts = path.stem.split("_")
    language_code, model = parts[1], parts[2] # change to parts[2], parts[3] for testing with EXAMPLE files
    frame = pd.read_csv(path).assign(
        language=language_names[language_code],
        model=model,
    )
    frames.append(frame)

predictions = pd.concat(frames, ignore_index=True)
answer_pattern = re.compile(r"<answer>\s*(?:[1-4])\s*</answer>")
predictions["format_error"] = ~predictions["raw_output"].fillna("").str.contains(answer_pattern)

input_price = {"small": 0.02, "big": 0.40}
output_price = {"small": 0.04, "big": 0.40}
predictions["input_cost"] = (
    predictions["prompt_tokens"] * predictions["model"].map(input_price) / 1_000_000
)
predictions["output_cost"] = (
    predictions["completion_tokens"] * predictions["model"].map(output_price) / 1_000_000
)

overview = predictions.groupby(["language", "model"], as_index=False).agg(
    n=("question_id", "count")
)
overview

,language,model,n
0,English,big,3
1,English,small,3


## Model accuracy

Accuracy is reported with its sample standard deviation and a 95% Wilson confidence interval.


In [43]:
def wilson_interval(successes, trials, z=1.96):
    proportion = successes / trials
    denominator = 1 + z**2 / trials
    centre = (proportion + z**2 / (2 * trials)) / denominator
    margin = z * (proportion * (1 - proportion) / trials + z**2 / (4 * trials**2)) ** 0.5 / denominator
    return centre - margin, centre + margin

accuracy_rows = []
for (language, model), group in predictions.groupby(["language", "model"]):
    n = len(group)
    correct = int(group["correct"].sum())
    low, high = wilson_interval(correct, n)
    accuracy_rows.append({
        "language": language,
        "model": model,
        "n": n,
        "mean_accuracy": group["correct"].mean(),
        "sd_accuracy": group["correct"].std(ddof=1),
        "format_error_rate": group["format_error"].mean(),
        "95ci_accuracy": f"[{low:.1%}, {high:.1%}]",
    })

accuracy_table = pd.DataFrame(accuracy_rows)
accuracy_table.style.format({
    "mean_accuracy": "{:.1%}",
    "sd_accuracy": "{:.3f}",
    "format_error_rate": "{:.1%}",
})


,language,model,n,mean_accuracy,sd_accuracy,format_error_rate,95ci_accuracy
0,English,big,3,100.0%,0.000,0.0%,"[43.8%, 100.0%]"
1,English,small,3,66.7%,0.577,0.0%,"[20.8%, 93.9%]"


## Token usage and inference cost

Token usage and monetary cost are shown in separate tables.


In [44]:
token_rows = []
for (language, model), group in predictions.groupby(["language", "model"]):
    total_tokens = group["prompt_tokens"] + group["completion_tokens"]
    token_rows.append({
        "language": language,
        "model": model,
        "n": len(group),
        "mean_in_tokens": group["prompt_tokens"].mean(),
        "sd_in_tokens": group["prompt_tokens"].std(ddof=1),
        "mean_out_tokens": group["completion_tokens"].mean(),
        "sd_out_tokens": group["completion_tokens"].std(ddof=1),
        "mean_total_tokens": total_tokens.mean(),
        "sd_total_tokens": total_tokens.std(ddof=1),
    })

token_table = pd.DataFrame(token_rows)
display( token_table.style.format({
    "mean_in_tokens": "{:.1f}",
    "sd_in_tokens": "{:.1f}",
    "mean_out_tokens": "{:.1f}",
    "sd_out_tokens": "{:.1f}",
    "mean_total_tokens": "{:.1f}",
    "sd_total_tokens": "{:.1f}",
}))

cost_rows = []
for (language, model), group in predictions.groupby(["language", "model"]):
    cost_rows.append({
        "language": language,
        "model": model,
        "n": len(group),
        "mean_in_cost": group["input_cost"].mean(),
        "sd_in_cost": group["input_cost"].std(ddof=1),
        "mean_out_cost": group["output_cost"].mean(),
        "sd_out_cost": group["output_cost"].std(ddof=1),
        "mean_total_cost": (group["input_cost"] + group["output_cost"]).mean(),
        "sd_total_cost": (group["input_cost"] + group["output_cost"]).std(ddof=1),
    })

cost_table = pd.DataFrame(cost_rows)
display(cost_table.style.format({
    "mean_in_cost": "${:.6f}",
    "sd_in_cost": "${:.6f}",
    "mean_out_cost": "${:.6f}",
    "sd_out_cost": "${:.6f}",
    "mean_total_cost": "${:.6f}",
    "sd_total_cost": "${:.6f}",
}))


,language,model,n,mean_in_tokens,sd_in_tokens,mean_out_tokens,sd_out_tokens,mean_total_tokens,sd_total_tokens
0,English,big,3,227.3,23.9,207.3,146.2,434.7,122.4
1,English,small,3,227.3,23.9,207.3,146.2,434.7,122.4


,language,model,n,mean_in_cost,sd_in_cost,mean_out_cost,sd_out_cost,mean_total_cost,sd_total_cost
0,English,big,3,$0.000091,$0.000010,$0.000083,$0.000058,$0.000174,$0.000049
1,English,small,3,$0.000005,$0.000000,$0.000008,$0.000006,$0.000013,$0.000005
